#  Spaceship Titanic — Advanced Kaggle Solution
## High Accuracy Ensemble Pipeline with Feature Engineering

Author: Shivam SIngh  
Competition: Spaceship Titanic  
Task: Binary Classification  
Platform: Kaggle  

---

# Introduction

The Spaceship Titanic competition is a machine learning classification challenge where the objective is to predict whether passengers were transported to an alternate dimension during a mysterious anomaly.

This notebook includes:

✅ Advanced Feature Engineering  
✅ Strong Exploratory Data Analysis  
✅ High Accuracy Ensemble Models  
✅ Cross Validation  
✅ Hyperparameter Tuning  
✅ Beautiful Visualizations  
✅ Kaggle Submission Pipeline  


In [ ]:

# =========================================================
# IMPORT LIBRARIES
# =========================================================

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score

from sklearn.preprocessing import LabelEncoder

from sklearn.impute import SimpleImputer

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import VotingClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# =========================================================
# LOAD DATASET
# =========================================================

train_df = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/train.csv')

print("Train Shape :", train_df.shape)
print("Test Shape :", test_df.shape)

In [ ]:
train_df.head()

In [ ]:
train_df.info()

In [ ]:
train_df.describe()

In [ ]:
plt.figure(figsize=(6,5))

sns.countplot(
    data=train_df,
    x='Transported'
)

plt.title('Transported Distribution')
plt.show()

In [ ]:
missing = train_df.isnull().sum().sort_values(ascending=False)

missing[missing > 0]

In [ ]:
plt.figure(figsize=(12,6))

sns.heatmap(train_df.isnull(), cbar=False)

plt.title("Missing Values Heatmap")
plt.show()

In [ ]:
numerical_cols = [
    'Age',
    'RoomService',
    'FoodCourt',
    'ShoppingMall',
    'Spa',
    'VRDeck'
]

fig, axes = plt.subplots(2,3, figsize=(18,10))

axes = axes.flatten()

for idx, col in enumerate(numerical_cols):

    sns.histplot(
        train_df[col],
        kde=True,
        ax=axes[idx]
    )

    axes[idx].set_title(f'{col} Distribution')

plt.tight_layout()
plt.show()

In [ ]:
for df in [train_df, test_df]:

    df[['Deck','CabinNum','Side']] = df['Cabin'].str.split('/', expand=True)

In [ ]:
for df in [train_df, test_df]:

    df['Group'] = df['PassengerId'].apply(lambda x: x.split('_')[0])

    df['GroupSize'] = df.groupby('Group')['Group'].transform('count')

In [ ]:
spending_features = [
    'RoomService',
    'FoodCourt',
    'ShoppingMall',
    'Spa',
    'VRDeck'
]

for df in [train_df, test_df]:

    df['TotalSpend'] = df[spending_features].sum(axis=1)

    df['NoSpending'] = (df['TotalSpend'] == 0).astype(int)

In [ ]:
for df in [train_df, test_df]:

    df['AgeGroup'] = pd.cut(
        df['Age'],
        bins=[0,12,18,25,40,60,100],
        labels=[
            'Child',
            'Teen',
            'Young',
            'Adult',
            'Middle',
            'Senior'
        ]
    )

In [ ]:
for df in [train_df, test_df]:

    df['LuxuryPassenger'] = (
        (df['Spa'].fillna(0) > 0) |
        (df['VRDeck'].fillna(0) > 0)
    ).astype(int)

In [ ]:
plt.figure(figsize=(12,8))

corr = train_df.select_dtypes(include=np.number).corr()

sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm'
)

plt.title('Correlation Heatmap')

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(
    data=train_df,
    x='HomePlanet',
    hue='Transported'
)

plt.title('HomePlanet vs Transported')

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

sns.boxplot(
    data=train_df,
    x='Transported',
    y='TotalSpend'
)

plt.title('Total Spending vs Transported')

plt.show()

In [ ]:
DROP_COLS = [
    'PassengerId',
    'Name',
    'Cabin'
]

train_df.drop(columns=DROP_COLS, inplace=True)
test_df.drop(columns=DROP_COLS, inplace=True)

In [ ]:
X = train_df.drop('Transported', axis=1)

y = train_df['Transported'].astype(int)

In [ ]:
categorical_cols = X.select_dtypes(include='object').columns

for col in categorical_cols:

    encoder = LabelEncoder()

    combined = pd.concat([
        X[col],
        test_df[col]
    ]).astype(str)

    encoder.fit(combined)

    X[col] = encoder.transform(X[col].astype(str))

    test_df[col] = encoder.transform(test_df[col].astype(str))

In [ ]:
# =========================================================
# HANDLE MISSING VALUES
# =========================================================

from sklearn.impute import SimpleImputer

# Separate numerical and categorical columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object', 'category']).columns

# =========================================================
# NUMERICAL IMPUTATION
# =========================================================

num_imputer = SimpleImputer(strategy='median')

X[num_cols] = num_imputer.fit_transform(X[num_cols])

test_df[num_cols] = num_imputer.transform(test_df[num_cols])

# =========================================================
# CATEGORICAL IMPUTATION
# =========================================================

cat_imputer = SimpleImputer(strategy='most_frequent')

X[cat_cols] = cat_imputer.fit_transform(X[cat_cols])

test_df[cat_cols] = cat_imputer.transform(test_df[cat_cols])

# =========================================================
# CHECK REMAINING NULL VALUES
# =========================================================

print("Missing Values in Train :", X.isnull().sum().sum())
print("Missing Values in Test  :", test_df.isnull().sum().sum())

In [ ]:
# =========================================================
# ENCODE CATEGORICAL FEATURES
# =========================================================

from sklearn.preprocessing import LabelEncoder

categorical_cols = X.select_dtypes(include=['object', 'category']).columns

for col in categorical_cols:

    encoder = LabelEncoder()

    combined_data = pd.concat([
        X[col],
        test_df[col]
    ]).astype(str)

    encoder.fit(combined_data)

    X[col] = encoder.transform(X[col].astype(str))

    test_df[col] = encoder.transform(test_df[col].astype(str))

print("Categorical Encoding Completed")

In [ ]:
# =========================================================
# TRAIN VALIDATION SPLIT
# =========================================================

from sklearn.model_selection import train_test_split

y = train_df['Transported'].astype(int)

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("X_train Shape :", X_train.shape)
print("X_valid Shape :", X_valid.shape)

In [ ]:
# =========================================================
# TRAIN CATBOOST MODEL
# =========================================================

from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(

    iterations=3000,
    learning_rate=0.03,
    depth=8,

    loss_function='Logloss',
    eval_metric='Accuracy',

    random_seed=42,
    verbose=200
)

cat_model.fit(
    X_train,
    y_train
)

In [ ]:
# =========================================================
# VALIDATION PREDICTIONS
# =========================================================

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

preds = cat_model.predict(X_valid)

accuracy = accuracy_score(
    y_valid,
    preds
)

print("Validation Accuracy :", accuracy)

In [ ]:
# =========================================================
# CLASSIFICATION REPORT
# =========================================================

print(
    classification_report(
        y_valid,
        preds
    )
)

In [ ]:
# =========================================================
# CONFUSION MATRIX
# =========================================================

import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(
    y_valid,
    preds
)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d'
)

plt.title("Confusion Matrix")

plt.show()

In [ ]:
# =========================================================
# FEATURE IMPORTANCE
# =========================================================

feature_importance = pd.DataFrame({

    'Feature': X.columns,

    'Importance': cat_model.feature_importances_

})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

plt.figure(figsize=(12,8))

sns.barplot(

    data=feature_importance.head(15),

    x='Importance',

    y='Feature'
)

plt.title("Top Important Features")

plt.show()

In [ ]:
# =========================================================
# TRAIN ON FULL DATASET
# =========================================================

cat_model.fit(X, y)

In [ ]:
# =========================================================
# RELOAD ORIGINAL TEST DATA
# =========================================================

test_df = pd.read_csv(
    '/kaggle/input/competitions/spaceship-titanic/test.csv'
)

# =========================================================
# SAVE PASSENGER IDS
# =========================================================

test_ids = test_df['PassengerId']

# =========================================================
# FEATURE ENGINEERING
# =========================================================

test_df[['Deck','CabinNum','Side']] = test_df['Cabin'].str.split('/', expand=True)

test_df['Group'] = test_df['PassengerId'].apply(
    lambda x: x.split('_')[0]
)

test_df['GroupSize'] = test_df.groupby('Group')['Group'].transform('count')

spending_features = [
    'RoomService',
    'FoodCourt',
    'ShoppingMall',
    'Spa',
    'VRDeck'
]

test_df['TotalSpend'] = test_df[spending_features].sum(axis=1)

test_df['NoSpending'] = (
    test_df['TotalSpend'] == 0
).astype(int)

test_df['LuxuryPassenger'] = (
    (test_df['Spa'].fillna(0) > 0) |
    (test_df['VRDeck'].fillna(0) > 0)
).astype(int)

test_df['AgeGroup'] = pd.cut(
    test_df['Age'],
    bins=[0,12,18,25,40,60,100],
    labels=[
        'Child',
        'Teen',
        'Young',
        'Adult',
        'Middle',
        'Senior'
    ]
)

# =========================================================
# DROP UNUSED COLUMNS
# =========================================================

DROP_COLS = [
    'PassengerId',
    'Name',
    'Cabin'
]

test_df.drop(columns=DROP_COLS, inplace=True)

# =========================================================
# CONVERT AGEGROUP TO STRING
# =========================================================

test_df['AgeGroup'] = test_df['AgeGroup'].astype(str)

# =========================================================
# MATCH TRAINING COLUMNS
# =========================================================

for col in X.columns:

    if col not in test_df.columns:

        test_df[col] = np.nan

# =========================================================
# REORDER COLUMNS
# =========================================================

test_df = test_df[X.columns]

# =========================================================
# HANDLE MISSING VALUES
# =========================================================

for col in test_df.columns:

    if test_df[col].dtype == 'object':

        test_df[col] = test_df[col].fillna('Unknown')

    else:

        test_df[col] = test_df[col].fillna(
            test_df[col].median()
        )

# =========================================================
# ENCODE CATEGORICAL FEATURES
# =========================================================

for col in test_df.select_dtypes(include='object').columns:

    encoder = LabelEncoder()

    combined = pd.concat([
        X[col].astype(str),
        test_df[col].astype(str)
    ])

    encoder.fit(combined)

    test_df[col] = encoder.transform(
        test_df[col].astype(str)
    )

# =========================================================
# FINAL PREDICTIONS
# =========================================================

test_predictions = cat_model.predict(test_df)

print("Predictions Shape :", len(test_predictions))

In [ ]:
# =========================================================
# CREATE SUBMISSION FILE
# =========================================================

submission = pd.DataFrame({

    'PassengerId': test_ids,

    'Transported': test_predictions.astype(bool)

})

submission.to_csv(
    'submission.csv',
    index=False
)

print("submission.csv created successfully 🚀")

In [ ]:
submission.head()